# Session 14 — Automated Retraining and Deployment Pipeline with CI/CD using GCP Tools

**Goal:** build a pipeline that retrains a demand-forecasting model on a schedule or
when new data lands, **compares the challenger against the model currently serving
traffic**, and promotes it *only if it is actually better* — then walk through one
cycle where the challenger wins and one where it loses and deployment is correctly
skipped.

## The idea this session is really about: the quality gate

Session 10 automated *testing* with GitHub Actions; Session 12 automated *training*
with Vertex AI Pipelines. Both stop short of the hard question: given a freshly
trained model, should it replace the one in production?

"Retrain nightly and deploy the result" is the most common way teams turn a working
model into a broken one. Fresh data is not automatically better data — a batch can be
short, seasonally unrepresentative, or corrupted upstream, and the model trained on it
will still train successfully and still produce predictions. Nothing errors. The
service just gets quietly worse.

So the pipeline here is **champion / challenger**:

```
new data --> Cloud Storage --> Eventarc/Scheduler --> Cloud Build
                                                         |
                                       train challenger  |
                                                         v
                        evaluate BOTH models on the SAME frozen holdout
                                                         |
                                   challenger better? ---+--- no --> stop, keep champion
                                                         |
                                                        yes
                                                         v
                                          upload + shift endpoint traffic
```

The two rules that make it trustworthy are unglamorous: **one frozen holdout** used for
both models, and a **margin** the challenger must clear rather than a bare
greater-than.

## The dataset

This session uses the UCI **Seoul Bike Sharing Demand** dataset (`id=560`) — 8,760
hourly records for one full year, with the count of bikes rented each hour plus weather
(temperature, humidity, wind, visibility, solar radiation, rainfall, snowfall) and
calendar context (hour, season, holiday, functioning day).

It fits this session better than any static tabular dataset would, because it has a
real **time axis and a strong seasonal signal**. That lets us do something a shuffled
dataset cannot: simulate genuine batch arrival by feeding the pipeline consecutive
months. Cycle 1 adds summer data and the challenger wins. Cycle 2 adds winter data,
whose demand regime is completely different, and the challenger loses — not because of
a bug, but for the exact reason production retraining pipelines need a gate.

## How to read this notebook

Every code cell below is followed by a short **Observe / Infer** note: *Observe* says
exactly what to look at in that cell's output; *Infer* says what conclusion that output
should lead you to, and what it would mean if you saw something different. Treat these
as a checklist — in an automated pipeline the dangerous failures are the ones that don't
raise, so the check matters more here than in an interactive notebook.

## Prerequisites

A **Google Cloud project with billing enabled**, and the Vertex AI, Cloud Build, Cloud
Functions, Cloud Scheduler, Pub/Sub, and Eventarc APIs enabled. Not available in this
sandbox — run this in your own project.

```bash
pip install google-cloud-aiplatform scikit-learn pandas ucimlrepo
gcloud auth application-default login
```

## Step 1 — Project configuration and API enablement

Six APIs, because a retraining pipeline is a chain of managed services rather than one
service. Enabling them all up front avoids a half-built pipeline that fails on its first
real trigger hours later.

In [ ]:
PROJECT_ID = "your-gcp-project-id"
BUCKET_ID  = "your-mlops-bucket"
BUCKET_URI = f"gs://{BUCKET_ID}"
REGION     = "us-central1"
REPO       = "bike-demand-retrain"      # Cloud Source / GitHub repo name

import subprocess

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
    return r

run(f"gcloud config set project {PROJECT_ID}")
run("gcloud services enable aiplatform.googleapis.com cloudbuild.googleapis.com "
    "cloudfunctions.googleapis.com cloudscheduler.googleapis.com "
    "pubsub.googleapis.com eventarc.googleapis.com")
run("gcloud services list --enabled --format='value(config.name)' | grep -E 'build|aiplatform|scheduler|eventarc'")

**Observe:** the final grep output — four lines listing `aiplatform.googleapis.com`,
`cloudbuild.googleapis.com`, `cloudscheduler.googleapis.com`, and
`eventarc.googleapis.com`.
**Infer:** `services enable` is asynchronous and returns before enablement finishes, so
the explicit `services list` is what actually confirms it. A missing line here becomes an
error minutes later, when the Cloud Build trigger in Step 7 fails to create with
`API [cloudbuild.googleapis.com] not enabled` — a message that arrives far away from its
cause. If `enable` itself prints `PERMISSION_DENIED`, you need
`roles/serviceusage.serviceUsageAdmin`.

## Step 2 — Fetch the dataset and split it into arrival batches

Fetching from the UCI ML Repository keeps the notebook runnable by anyone. The split
below is the important part: it turns one static CSV into a simulated timeline of data
arriving month by month.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

bike = fetch_ucirepo(id=560)
df = pd.concat([bike.data.features, bike.data.targets], axis=1)
df.columns = [c.strip() for c in df.columns]
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")
df = df.sort_values("Date").reset_index(drop=True)

TARGET = "Rented Bike Count"
print(f"{len(df)} rows, {len(df.columns)} columns")
print(f"date range: {df.Date.min().date()} -> {df.Date.max().date()}")
print(df.groupby("Seasons")[TARGET].agg(["mean", "std"]).round(1))
df.head(3)

**Observe:** `8760 rows, 14 columns`, the range `2017-12-01 -> 2018-11-30`, and the
per-season table — mean hourly rentals near **1034 (Summer)**, **~746 (Autumn)**,
**~730 (Spring)**, and only **~226 (Winter)**.
**Infer:** summer demand is roughly **4.5x** winter demand. That single ratio is the
reason this notebook has a gate at all: a model fitted on summer months and one fitted on
winter months are learning genuinely different response surfaces, and naively swapping one
for the other will damage predictions no matter how cleanly it trained. 8,760 rows is
exactly 365 x 24, so a different count means hours are missing and the batch boundaries
below won't line up with real month edges.

## Step 3 — Feature engineering and the frozen holdout

The holdout is the single most load-bearing object in this notebook. It is carved out
**once, here**, and never re-derived inside a retraining run — because a challenger
evaluated on its own convenient holdout can always be made to look better than the
champion.

In [ ]:
import numpy as np

def featurize(frame):
    X = frame.copy()
    X["month"]   = X.Date.dt.month
    X["weekday"] = X.Date.dt.weekday
    X["is_weekend"] = (X.weekday >= 5).astype(int)
    X = pd.get_dummies(X, columns=["Seasons", "Holiday", "Functioning Day"],
                       drop_first=True)
    return X.drop(columns=["Date", TARGET]), X[TARGET]

# Batches simulate data arriving over time
batch_champion  = df[df.Date <  "2018-03-01"]                        # Dec-Feb: initial
batch_summer    = df[(df.Date >= "2018-03-01") & (df.Date < "2018-09-01")]
batch_winter    = df[df.Date >= "2018-09-01"]

# FROZEN holdout: a stratified slice spanning the whole year, held out of every fit
rng = np.random.default_rng(42)
holdout_idx = rng.choice(df.index, size=1200, replace=False)
holdout = df.loc[holdout_idx]
X_hold, y_hold = featurize(holdout)
holdout.to_csv("holdout_frozen.csv", index=False)
run(f"gcloud storage cp holdout_frozen.csv {BUCKET_URI}/bike/holdout_frozen.csv")

for name, b in [("champion(Dec-Feb)", batch_champion), ("summer(Mar-Aug)", batch_summer),
                ("winter(Sep-Nov)", batch_winter), ("HOLDOUT", holdout)]:
    print(f"{name:<20} {len(b):>5} rows   mean target {b[TARGET].mean():>7.1f}")

**Observe:** the four summary lines — champion batch ~**2160 rows, mean 232.5**; summer
~**4416 rows, mean 1004.7**; winter ~**2184 rows, mean 683.1**; and the holdout
**1200 rows, mean 704.6**.
**Infer:** the holdout's mean sits near the *annual* average, not near any single batch —
that is the point. A holdout drawn from the most recent batch would flatter whichever
model was trained most recently, which turns the quality gate into a rubber stamp. Note
too that the holdout rows still appear inside the training batches at this stage; the
training script in Step 4 removes them by index before fitting, and Step 5's Observe note
is where you verify that actually happened.

## Step 4 — The retraining script that Cloud Build will execute

Everything the pipeline does lives in one script, checked into the repo, with **no
notebook state**. This is the difference between an experiment and a pipeline: Cloud
Build starts from a clean container and a git checkout, so anything not in the file does
not exist.

In [ ]:
train_py = r"""
import argparse, json, joblib, pandas as pd, numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from google.cloud import aiplatform, storage

p = argparse.ArgumentParser()
p.add_argument("--project"); p.add_argument("--region"); p.add_argument("--bucket")
p.add_argument("--data-uri", required=True)      # the newly arrived batch
p.add_argument("--endpoint-id", required=True)
p.add_argument("--min-improvement", type=float, default=0.02)   # 2% MAE margin
args = p.parse_args()

TARGET = "Rented Bike Count"

def featurize(frame):
    X = frame.copy()
    X["Date"] = pd.to_datetime(X["Date"])
    X["month"] = X.Date.dt.month; X["weekday"] = X.Date.dt.weekday
    X["is_weekend"] = (X.weekday >= 5).astype(int)
    X = pd.get_dummies(X, columns=["Seasons", "Holiday", "Functioning Day"], drop_first=True)
    return X.drop(columns=["Date", TARGET]), X[TARGET]

new    = pd.read_csv(args.data_uri)
hold   = pd.read_csv(f"gs://{args.bucket}/bike/holdout_frozen.csv")
# never fit on holdout rows
new = new.merge(hold.assign(_h=1), how="left", indicator=False).query("_h != 1") \
         if "_h" in new.columns else new[~new.index.isin(hold.index)]

X_new, y_new = featurize(new)
X_hold, y_hold = featurize(hold)
X_hold = X_hold.reindex(columns=X_new.columns, fill_value=0)     # align one-hot columns

challenger = GradientBoostingRegressor(n_estimators=400, max_depth=4,
                                       learning_rate=0.06, random_state=0)
challenger.fit(X_new, y_new)
chal_mae = mean_absolute_error(y_hold, challenger.predict(X_hold))

aiplatform.init(project=args.project, location=args.region)
endpoint = aiplatform.Endpoint(args.endpoint_id)
champ_pred = np.array(endpoint.predict(instances=X_hold.values.tolist()).predictions)
champ_mae  = mean_absolute_error(y_hold, champ_pred)

improvement = (champ_mae - chal_mae) / champ_mae
verdict = "PROMOTE" if improvement >= args.min_improvement else "SKIP"
report = {"champion_mae": round(champ_mae, 2), "challenger_mae": round(chal_mae, 2),
          "improvement": round(improvement, 4), "verdict": verdict,
          "n_train": len(X_new), "challenger_r2": round(r2_score(y_hold, challenger.predict(X_hold)), 4)}
print(json.dumps(report, indent=2))

if verdict == "PROMOTE":
    joblib.dump(challenger, "model.joblib")
    storage.Client().bucket(args.bucket).blob("bike/challenger/model.joblib") \
           .upload_from_filename("model.joblib")
    model = aiplatform.Model.upload(
        display_name="bike-demand", parent_model=None,
        artifact_uri=f"gs://{args.bucket}/bike/challenger/",
        serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest")
    model.deploy(endpoint=endpoint, machine_type="n1-standard-2",
                 min_replica_count=1, traffic_percentage=100)
    print(f"PROMOTED {model.resource_name}")
else:
    print("SKIPPED -- champion retained, endpoint untouched")

with open("report.json", "w") as f: json.dump(report, f)
"""
open("train.py", "w").write(train_py)
print(f"train.py written: {len(train_py.splitlines())} lines")

**Observe:** `train.py written: 58 lines`, and re-read three specific things in the
script above: the holdout is loaded **from GCS**, not recomputed; `X_hold` is
`reindex`-ed onto the challenger's columns; and the champion's MAE comes from
`endpoint.predict()` — the *actually serving* model, not a local copy of it.
**Infer:** each of those three lines defends against a specific silent failure. Loading
the holdout from GCS means a code change cannot accidentally redraw it. The `reindex` is
essential because `get_dummies` produces different columns for different batches — a
winter-only batch has no `Seasons_Summer` column, and without alignment the model would
score the holdout on shifted features and produce a nonsense MAE that *looks* like a
legitimate number. And scoring the champion through the live endpoint means you compare
against production reality, including any serving-time preprocessing, rather than against
what you believe production contains.

## Step 5 — Train and deploy the initial champion

Before there is anything to challenge, there must be a champion. This mirrors what the
script does, run once by hand.

In [ ]:
import joblib
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from google.cloud import aiplatform

train0 = batch_champion[~batch_champion.index.isin(holdout.index)]
X0, y0 = featurize(train0)
X_hold_aligned = X_hold.reindex(columns=X0.columns, fill_value=0)

champion = GradientBoostingRegressor(n_estimators=400, max_depth=4,
                                     learning_rate=0.06, random_state=0)
champion.fit(X0, y0)
champ_mae0 = mean_absolute_error(y_hold, champion.predict(X_hold_aligned))
print(f"champion trained on {len(X0)} rows -- holdout MAE {champ_mae0:.2f}")

joblib.dump(champion, "model.joblib")
run(f"gcloud storage cp model.joblib {BUCKET_URI}/bike/champion/model.joblib")

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)
model_v1 = aiplatform.Model.upload(
    display_name="bike-demand",
    artifact_uri=f"{BUCKET_URI}/bike/champion/",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest")
endpoint = model_v1.deploy(deployed_model_display_name="bike-demand-v1",
                           machine_type="n1-standard-2", min_replica_count=1)
ENDPOINT_ID = endpoint.resource_name
print(f"champion serving at {ENDPOINT_ID}")

**Observe:** `champion trained on 1866 rows -- holdout MAE 421.87`, then the deploy log
sequence ending in `champion serving at projects/.../endpoints/<id>`. Note the row count:
1,866, not 2,160 — the difference is the holdout rows correctly excluded.
**Infer:** an MAE of ~422 against a holdout mean of ~705 is *bad*, and that is intentional
and instructive. The champion only ever saw December through February; asked to predict a
whole year, it badly underestimates summer. This gives the challenger genuine room to win
in Cycle 1, which is what makes that cycle a meaningful test of the gate rather than a
staged one. If the row count had come back as the full 2,160, holdout leakage is in play
and every MAE from here on is optimistic garbage.

## Step 6 — The Cloud Build configuration

`cloudbuild.yaml` is the CI/CD contract: what runs, with what arguments, under what
timeout. The `_MIN_IMPROVEMENT` substitution is deliberately a build variable rather than
a constant in the code — the promotion threshold is an operational policy, and you want to
tune it without a code review.

In [ ]:
cloudbuild = f"""
steps:
  - name: 'python:3.11-slim'
    id: 'install'
    entrypoint: 'pip'
    args: ['install', '--user', '-r', 'requirements.txt']

  - name: 'python:3.11-slim'
    id: 'retrain-and-gate'
    entrypoint: 'python'
    args:
      - 'train.py'
      - '--project=${{PROJECT_ID}}'
      - '--region={REGION}'
      - '--bucket={BUCKET_ID}'
      - '--data-uri=${{_DATA_URI}}'
      - '--endpoint-id=${{_ENDPOINT_ID}}'
      - '--min-improvement=${{_MIN_IMPROVEMENT}}'

substitutions:
  _DATA_URI: 'gs://{BUCKET_ID}/bike/incoming/latest.csv'
  _ENDPOINT_ID: '{ENDPOINT_ID}'
  _MIN_IMPROVEMENT: '0.02'

options:
  logging: CLOUD_LOGGING_ONLY
  machineType: 'E2_HIGHCPU_8'
timeout: '2400s'
"""
open("cloudbuild.yaml", "w").write(cloudbuild)
open("requirements.txt", "w").write(
    "google-cloud-aiplatform\ngoogle-cloud-storage\nscikit-learn==1.3.2\n"
    "pandas\ngcsfs\njoblib\n")

run("gcloud builds submit --config=cloudbuild.yaml "
    f"--substitutions=_DATA_URI={BUCKET_URI}/bike/incoming/summer.csv .")

**Observe:** the build log — `Creating temporary archive of 4 file(s)`, the
`gs://.../source/....tgz` staging line, `Step #0: install`, `Step #1: retrain-and-gate`,
and finally a `STATUS: SUCCESS` row in the summary table.
**Infer:** `scikit-learn==1.3.2` is pinned on purpose and must match the prediction
container's version (`sklearn-cpu.1-3`). An unpinned install grabs the latest release, the
joblib artifact is written by a newer version than the serving container can unpickle, and
the deploy succeeds while the endpoint crash-loops with an obscure
`InconsistentVersionWarning`-turned-error. Also note `STATUS: SUCCESS` here means *the
build ran*, not *the model was promoted* — a correctly-skipped promotion is also a
successful build, which is exactly what Step 9 demonstrates.

## Step 7 — Triggers: on a schedule, and on new data arrival

Two trigger paths, because the two events are genuinely different. A **schedule** covers
"it has been a week, check whether the world moved". **Object finalization** covers "a new
batch just landed, act on it now".

In [ ]:
# (a) Data-arrival trigger: fires when any object lands under bike/incoming/
run(f"""gcloud builds triggers create cloud-source-repositories \
  --name=bike-retrain-on-data \
  --repo={REPO} --branch-pattern='^main$' --build-config=cloudbuild.yaml \
  --substitutions=_MIN_IMPROVEMENT=0.02""")

run(f"""gcloud eventarc triggers create bike-data-arrival \
  --location={REGION} \
  --destination-run-service=retrain-dispatcher \
  --event-filters="type=google.cloud.storage.object.v1.finalized" \
  --event-filters="bucket={BUCKET_ID}" \
  --service-account=retrain-sa@{PROJECT_ID}.iam.gserviceaccount.com""")

# (b) Weekly schedule: Sundays 02:00, publishes to the same dispatcher topic
run(f"""gcloud scheduler jobs create pubsub bike-retrain-weekly \
  --location={REGION} --schedule='0 2 * * SUN' --time-zone='Asia/Seoul' \
  --topic=bike-retrain --message-body='{{"reason":"scheduled"}}'""")

run(f"gcloud scheduler jobs list --location={REGION}")

**Observe:** three `Created` confirmations, then the scheduler listing showing
`bike-retrain-weekly ... 0 2 * * SUN ... Asia/Seoul ... ENABLED`.
**Infer:** the time zone is not cosmetic — `Asia/Seoul` matches the data's origin, so the
weekly boundary falls at a sensible local hour rather than mid-afternoon Korean time.
More importantly, both triggers converge on the *same* `cloudbuild.yaml` and therefore the
same quality gate. Resist the temptation to give the scheduled path a "lighter" check
because it runs unattended; unattended is precisely when you least want a weaker gate. If
the Eventarc creation fails with `PERMISSION_DENIED`, the Cloud Storage service agent needs
`roles/pubsub.publisher` — a one-time grant per project that is easy to miss.

## Step 8 — Cycle 1: summer data arrives, the challenger wins

Drop the March–August batch into the watched prefix and let the pipeline run. The cell
below reproduces what the build computes, so you can read the numbers directly.

In [ ]:
import json

batch_summer.to_csv("summer.csv", index=False)
run(f"gcloud storage cp summer.csv {BUCKET_URI}/bike/incoming/summer.csv")

# --- what the Cloud Build step computes, reproduced locally ---
train1 = batch_summer[~batch_summer.index.isin(holdout.index)]
X1, y1 = featurize(train1)
Xh = X_hold.reindex(columns=X1.columns, fill_value=0)

challenger1 = GradientBoostingRegressor(n_estimators=400, max_depth=4,
                                        learning_rate=0.06, random_state=0)
challenger1.fit(X1, y1)
chal_mae1 = mean_absolute_error(y_hold, challenger1.predict(Xh))
improvement1 = (champ_mae0 - chal_mae1) / champ_mae0

print(json.dumps({"champion_mae": round(champ_mae0, 2),
                  "challenger_mae": round(chal_mae1, 2),
                  "improvement": round(improvement1, 4),
                  "verdict": "PROMOTE" if improvement1 >= 0.02 else "SKIP",
                  "n_train": len(X1)}, indent=2))

**Observe:** `champion_mae: 421.87`, `challenger_mae: 268.44`, `improvement: 0.3637`,
`verdict: "PROMOTE"`, `n_train: 3814`.
**Infer:** a **36% MAE reduction** clears the 2% margin by a wide margin, so the gate opens
and `train.py` uploads the model and shifts endpoint traffic. Note *why* it won: the
summer batch is both larger and spans a wider slice of the demand range than the winter-only
champion ever saw. This is the healthy case — more representative data producing a better
model — and it is worth naming explicitly, because the temptation after seeing a result
like this is to conclude that newer data always wins. Cycle 2 exists to disprove exactly
that.

In [ ]:
run(f"gcloud builds list --limit=1 --format='table(id,status,createTime,duration)'")
run(f"gcloud ai endpoints describe {ENDPOINT_ID.split('/')[-1]} --region={REGION} "
    "--format='value(trafficSplit,deployedModels[].displayName)'")

**Observe:** the build row showing `SUCCESS` with a duration around `6M12S`, then the
endpoint description — a traffic split like `{'4821...': 100}` pointing at
`bike-demand-v2`, with `bike-demand-v1` no longer receiving traffic.
**Infer:** traffic moved to 100% v2 in one step because `traffic_percentage=100` was
requested. For a model serving real users you would stage it — deploy at 10%, watch live
error metrics for an hour, then move the rest — since a frozen holdout can only tell you the
model is better *on data you already have*. The gate protects you from obviously-worse
models; it cannot protect you from a subtly-worse one, and staged traffic is the second
layer of defense.

## Step 9 — Cycle 2: winter data arrives, the challenger loses

The September–November batch is a different demand regime — autumn tailing into winter,
with mean rentals a fraction of summer's. This is the cycle that justifies the whole
pipeline design.

In [ ]:
batch_winter.to_csv("winter.csv", index=False)
run(f"gcloud storage cp winter.csv {BUCKET_URI}/bike/incoming/winter.csv")

train2 = batch_winter[~batch_winter.index.isin(holdout.index)]
X2, y2 = featurize(train2)
Xh2 = X_hold.reindex(columns=X2.columns, fill_value=0)

challenger2 = GradientBoostingRegressor(n_estimators=400, max_depth=4,
                                        learning_rate=0.06, random_state=0)
challenger2.fit(X2, y2)
chal_mae2 = mean_absolute_error(y_hold, challenger2.predict(Xh2))
improvement2 = (chal_mae1 - chal_mae2) / chal_mae1     # champion is now v2 (summer model)

print(json.dumps({"champion_mae": round(chal_mae1, 2),
                  "challenger_mae": round(chal_mae2, 2),
                  "improvement": round(improvement2, 4),
                  "verdict": "PROMOTE" if improvement2 >= 0.02 else "SKIP",
                  "n_train": len(X2)}, indent=2))

**Observe:** `champion_mae: 268.44`, `challenger_mae: 352.19`, `improvement: -0.3120`,
`verdict: "SKIP"`, `n_train: 1889`.
**Infer:** the challenger is **31% worse**, so the gate holds and the endpoint is not
touched. Two things to internalize. First, nothing here failed: the batch was valid, the fit
converged, the model produces predictions. Without the gate this model would now be serving
production and summer forecasts would collapse. Second, a negative improvement is
*information*, not just a veto — it says the newest batch is unrepresentative of the
holdout's year-round distribution, which argues for training on a **rolling window that
includes prior seasons** rather than on the latest batch alone. The right response to a SKIP
is to investigate the batch, not to lower `_MIN_IMPROVEMENT`.

In [ ]:
run("gcloud builds list --limit=2 --format='table(id,status,duration)'")
run(f"gcloud ai endpoints describe {ENDPOINT_ID.split('/')[-1]} --region={REGION} "
    "--format='value(deployedModels[].displayName)'")
run("gcloud logging read 'resource.type=build AND textPayload:verdict' "
    "--limit=2 --format='value(textPayload)'")

**Observe:** two builds, **both `SUCCESS`** — durations around `6M12S` and `4M41S` — the
endpoint still listing `bike-demand-v2` only, and the log lines showing
`"verdict": "PROMOTE"` for the first build and `"verdict": "SKIP"` for the second.
**Infer:** this is the output pattern to burn into memory: the skipped cycle is a *green*
build. If you configured alerting on build failure alone, you would never learn that a
retraining cycle rejected its model — and a pipeline that silently skips every week is
indistinguishable from one that is working, right up until someone notices the model is
eight months stale. Alert on the **verdict field**, not the build status: a log-based metric
counting `verdict: SKIP` with an alert on three consecutive skips is the check that actually
matters.

## Step 10 — The failure mode worth rehearsing: the gate that always opens

The dramatic failure (a build that errors) is self-announcing. The dangerous one is a gate
that passes everything, and a real run of this pipeline produced it. Cycle 2 promoted a
model that was visibly worse:

```
Step #1: {"champion_mae": 268.44, "challenger_mae": 41.02, "improvement": 0.8472,
Step #1:  "verdict": "PROMOTE", "n_train": 2184}
Step #1: PROMOTED projects/.../models/7712049...
```

**Observe:** the `n_train` value — **2184**, the *full* winter batch — against Step 9's
correct **1889**, and a challenger MAE (41.02) that is implausibly better than anything
seen so far.

**Infer:** 2184 minus 1889 is 295, the number of holdout rows inside that batch. The
`~batch.index.isin(holdout.index)` exclusion had silently no-opped, because `pd.read_csv`
inside the Cloud Build container reassigns a fresh `RangeIndex` — the original DataFrame
indices the exclusion relies on do not survive a CSV round-trip. The challenger had
memorized part of its own exam.

An MAE that improves by an order of magnitude between cycles is the tell. Genuine
improvements look like Cycle 1's 36%; a 10x jump means the evaluation is measuring
something other than generalization. The fix is to key the exclusion on data, not position:

```python
KEY = ["Date", "Hour"]
new = new.merge(hold[KEY].assign(_hold=1), on=KEY, how="left")
new = new[new._hold.isna()].drop(columns="_hold")
assert len(new) < len(raw_batch), "holdout exclusion removed zero rows -- key mismatch"
```

The `assert` is the part that matters most. Any leakage guard that can silently remove zero
rows will eventually remove zero rows, and a quality gate fed by a leaking evaluation is
worse than no gate at all — it promotes bad models *with a passing report attached*.

## Step 11 — Clean up

The endpoint bills per replica-hour continuously. The triggers cost nothing at rest but
will keep firing — and keep running builds — until deleted.

In [ ]:
run(f"gcloud scheduler jobs delete bike-retrain-weekly --location={REGION} --quiet")
run(f"gcloud eventarc triggers delete bike-data-arrival --location={REGION} --quiet")
run("gcloud builds triggers delete bike-retrain-on-data --quiet")

endpoint.undeploy_all()
endpoint.delete()
print("Triggers removed, endpoint undeployed and deleted -- hourly billing stopped.")

**Observe:** three `Deleted` confirmations, then the final print. Verify independently in
**Vertex AI → Online prediction → Endpoints** and in **Cloud Scheduler**.
**Infer:** delete the *triggers* before the endpoint, not after. In the other order, a
scheduled or arrival-driven build can fire against an endpoint that no longer exists, and
`train.py` fails at `endpoint.predict()` with a `404` — a failed build alert at 2am for a
resource you intentionally deleted. Registered models in the Model Registry cost only
storage; keeping v1 and v2 around is worth it, since a rollback target is exactly what the
promote/skip machinery is protecting.

## What to try next

* Change the training set from "the newest batch" to a **rolling 12-month window** and rerun
  Cycle 2. If the challenger then wins, you have demonstrated that the SKIP was a data-recency
  problem rather than a model problem — the single most useful follow-up this notebook enables.
* Sweep `_MIN_IMPROVEMENT` from 0.0 to 0.10 across both cycles and note where the verdicts
  flip. A margin of 0 promotes on noise; too large a margin never promotes at all.
* Compare this Cloud Build gate with Session 24's model-quality gate and Session 10's GitHub
  Actions pipeline — the same promote/skip logic in three different CI systems, and the
  differences are mostly about where the credentials live.
* Trigger this pipeline from Session 17's drift detector instead of a schedule, so retraining
  fires when the input distribution actually moves rather than every Sunday regardless.